## nb_inference

**Hourly inference** — reads the last 6 hours from `live_feed/rolling_buffer/` (actuals)
plus the next 8 hours from `live_feed/class_a_schedule/` (future schedule), builds the
6-snapshot temporal graph sequence, runs the champion `Seq2SeqGNN`, and writes 350 rows
(70 airports × 5 horizons) to two Delta tables:

- `predictions_latest` (overwritten each run) — DirectLake source for Power BI
- `predictions_history` (append-only) — audit + trend lines

**Trigger:** `pl_hourly_predict` fires at `:15` UTC each hour, 10 minutes after
`pl_fake_ingestion` writes the new buffer partition.

**One-time setup before first run:**
1. Build the wheel locally: `python -m build --wheel` → `dist/flight_delay_propagation-0.1.0-py3-none-any.whl`
2. Upload the wheel to OneLake → `Files/packages/`
3. Attach `FlightData_Lakehouse` as the default lakehouse for this notebook
4. (Recommended) Attach a Fabric custom environment `env_inference` with `torch>=2.1`, `torch-geometric>=2.4`, `deltalake>=0.14`, `pyarrow>=14` pre-installed — cuts cell 3 from ~5min cold-install down to ~10s.

In [ ]:
# Config — paths use the mounted lakehouse so no ADLS auth is needed.
HORIZONS           = [1, 2, 4, 6, 8]
INPUT_WINDOW       = 6      # snapshots fed to the seq2seq encoder
BUFFER_HOURS_READ  = 24     # read more than INPUT_WINDOW so we tolerate gaps + low-traffic hours
MIN_ROUTE_FLIGHTS  = 2      # training used 30 over 2 years; for inference, 2 is the equivalent noise filter
WINDOW_HOURS       = 1
DEVICE             = "cpu"  # F2 trial has no GPU; CPU inference is ~1–2s for 6 snapshots

LAKEHOUSE_FILES    = "/lakehouse/default/Files"
LAKEHOUSE_TABLES   = "/lakehouse/default/Tables"
MODEL_DIR          = f"{LAKEHOUSE_FILES}/models/champion"
BUFFER_ROOT        = f"{LAKEHOUSE_FILES}/live_feed/rolling_buffer"
SCHEDULE_ROOT      = f"{LAKEHOUSE_FILES}/live_feed/class_a_schedule"
PACKAGES_ROOT      = f"{LAKEHOUSE_FILES}/packages"

PREDICTIONS_LATEST  = f"{LAKEHOUSE_TABLES}/predictions_latest"
PREDICTIONS_HISTORY = f"{LAKEHOUSE_TABLES}/predictions_history"

In [ ]:
# Install the project wheel from the lakehouse. With env_inference attached,
# torch / torch-geometric / deltalake are already present and this finishes in ~10s.
import glob, subprocess, sys

wheels = sorted(glob.glob(f"{PACKAGES_ROOT}/flight_delay_propagation*.whl"))
if not wheels:
    raise FileNotFoundError(
        f"No wheel under {PACKAGES_ROOT}. Build locally with "
        f"'python -m build --wheel' and upload the .whl to Files/packages/."
    )

subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", wheels[-1]])
print(f"Installed: {wheels[-1].split('/')[-1]}")

In [ ]:
import json
from pathlib import Path

import pandas as pd
import torch
from deltalake import DeltaTable, write_deltalake

from src.data.graph_builder import (
    build_edge_index,
    create_temporal_graphs,
    WeatherLookups,
)
from src.models.factory import build_model
from src.utils.io import load_checkpoint_inference_only

In [ ]:
# Load the four champion artifacts uploaded in Phase 0d.
airport_map   = json.loads(Path(f"{MODEL_DIR}/airport_map.json").read_text())
feature_stats = torch.load(
    f"{MODEL_DIR}/feature_stats.pt", map_location=DEVICE, weights_only=True
)
metadata      = json.loads(Path(f"{MODEL_DIR}/metadata.json").read_text())

checkpoint_path = f"{MODEL_DIR}/{metadata['checkpoint_file']}"

print(f"Airports:   {len(airport_map)}")
print(f"Horizons:   {metadata.get('prediction_horizons', HORIZONS)}")
print(f"Checkpoint: {Path(checkpoint_path).name}")
print(f"Norm stats: mean shape={tuple(feature_stats['mean'].shape)}")

In [ ]:
# Read the last BUFFER_HOURS_READ partitions of rolling_buffer (actuals, CLASS_B_COLS).
# We read more than INPUT_WINDOW so that hours with <10 flights (graph_builder's
# minimum) don't cause us to drop below the seq2seq input length.
now_utc = pd.Timestamp.utcnow().floor("h")
frames_b, missing_b = [], []

for offset in range(BUFFER_HOURS_READ, 0, -1):
    ts = now_utc - pd.Timedelta(hours=offset)
    path = Path(BUFFER_ROOT) / f"{ts.year}/{ts.month:02d}/{ts.day:02d}/{ts.hour:02d}/flights.parquet"
    if path.exists():
        frames_b.append(pd.read_parquet(path))
    else:
        missing_b.append(ts)

if missing_b:
    print(f"WARNING: {len(missing_b)} buffer partitions missing")

if len(frames_b) < INPUT_WINDOW:
    raise RuntimeError(
        f"Only {len(frames_b)}/{INPUT_WINDOW} buffer partitions available. "
        f"Run nb_backfill_buffer first or wait for pl_fake_ingestion to catch up."
    )

df_buffer = pd.concat(frames_b, ignore_index=True)
print(f"Buffer: {len(df_buffer):,} rows across {len(frames_b)} partitions "
      f"({now_utc - pd.Timedelta(hours=BUFFER_HOURS_READ)} → {now_utc})")

In [ ]:
# Read the most recent class_a_schedule file. Each file contains 9 hours of
# forward schedule data INSIDE it (see GetFlightData function: it writes
# CRSDepTime in [H*100, (H+9)*100) under the path of the hour it ran).
# So a single recent file is enough — we don't iterate forward through hour
# paths because those paths don't exist (the function only writes at the
# timestamp it ran, not at each future hour the schedule covers).
df_schedule = pd.DataFrame()
loaded_from = None
for offset in range(0, 24):
    ts = now_utc - pd.Timedelta(hours=offset)
    path = Path(SCHEDULE_ROOT) / f"{ts.year}/{ts.month:02d}/{ts.day:02d}/{ts.hour:02d}/schedule.parquet"
    if path.exists():
        df_schedule = pd.read_parquet(path)
        loaded_from = ts
        break

if df_schedule.empty:
    print("WARNING: No class_a_schedule files found in the last 24h")
else:
    age_h = (now_utc - loaded_from).total_seconds() / 3600
    print(f"Schedule: {len(df_schedule):,} rows from {loaded_from} (T-{age_h:.0f}h, contains 9h of forward schedule)")

df_all = pd.concat([df_buffer, df_schedule], ignore_index=True)
print(f"Combined: {len(df_all):,} rows fed to the graph builder")

In [ ]:
# Build the airport graph (3 static edge features) and the snapshot list.
edge_index, edge_attr_static, edge_pairs = build_edge_index(
    df_all, airport_map, min_flights=MIN_ROUTE_FLIGHTS,
)
print(f"Edges: {edge_index.shape[1]} (min_flights={MIN_ROUTE_FLIGHTS})")

# Derive the expected feature dim from the saved normalization stats:
#   55  → checkpoint trained without weather block (weather.enabled=false)
#   109 → checkpoint trained with weather block (5 horizons, with 9 hist + 9*5 future cols)
# Passing WeatherLookups() (empty) produces 109-dim features with the
# weather block zero-filled; passing None produces 55-dim features.
# Either way, we match whatever the checkpoint was trained on.
expected_dim = feature_stats["mean"].shape[0]
empty_weather = WeatherLookups() if expected_dim == 109 else None
print(f"feature_stats expects input_dim={expected_dim} → weather block {'enabled (zero-fill)' if empty_weather else 'disabled'}")

graphs = create_temporal_graphs(
    df=df_all,
    airport_map=airport_map,
    edge_index=edge_index,
    edge_attr_static=edge_attr_static,
    edge_pairs=edge_pairs,
    window_hours=WINDOW_HOURS,
    prediction_horizons=HORIZONS,
    weather_lookups=empty_weather,
)

input_dim = graphs[0].x.shape[1] if graphs else None
print(f"Snapshots built: {len(graphs)} (input_dim={input_dim})")

if len(graphs) < INPUT_WINDOW:
    raise RuntimeError(
        f"Only {len(graphs)} snapshots formed; need at least {INPUT_WINDOW}. "
        f"Likely cause: rolling_buffer partitions had <10 flights each, "
        f"or a partition gap dropped too many candidates."
    )

In [ ]:
# Take the most recent INPUT_WINDOW snapshots and apply frozen normalization.
sequence = graphs[-INPUT_WINDOW:]
mean = feature_stats["mean"]
std  = feature_stats["std"]

for g in sequence:
    g.x = (g.x - mean) / std

print(f"Sequence length: {len(sequence)}")
print(f"First snapshot ends at: {sequence[0].timestamp}")
print(f"Last snapshot ends at:  {sequence[-1].timestamp}")

In [ ]:
# Build the model with the same dims as configs/weekend/seq2seq_gnn_large.yaml.
# loss="multi_task" forces the factory to wire output_channels=3 (arr_delay,
# dep_delay, pct_delayed); we only consume channels 0 and 2.
config = {
    "model": {
        "name": "seq2seq_gnn",
        "seq2seq_gnn": {
            "hidden_dim":          256,
            "num_heads":           8,
            "num_spatial_layers":  3,
            "num_temporal_layers": 2,
            "dropout":             0.3,
        },
    },
    "graph":    {"prediction_horizons": HORIZONS},
    "training": {"loss": "multi_task"},
}

model = build_model(config, input_dim=input_dim, edge_dim=5).to(DEVICE)
ckpt  = load_checkpoint_inference_only(checkpoint_path, model, device=DEVICE)
model.eval()

print(f"Loaded checkpoint epoch={ckpt['epoch']}")
print(f"Training metrics:   {ckpt.get('metrics', {})}")

In [ ]:
# Forward pass → [N, H, C] = [70, 5, 3]. Channel 0 = ArrDelay (min), channel 2 = P(delay≥15).
with torch.no_grad():
    preds = model(sequence)

arr_delay = preds[:, :, 0].cpu().numpy()
pct_15    = preds[:, :, 2].cpu().numpy().clip(0.0, 1.0)

idx_to_iata = {v: k for k, v in airport_map.items()}

rows = []
for node_idx in range(len(airport_map)):
    iata = idx_to_iata[node_idx]
    for h_idx, h in enumerate(HORIZONS):
        target_ts = now_utc + pd.Timedelta(hours=h)
        rows.append({
            "airport_code":            iata,
            "horizon_h":               int(h),
            "predicted_arr_delay_min": float(arr_delay[node_idx, h_idx]),
            "pct_arr_delayed_15":      float(pct_15[node_idx, h_idx]),
            "target_ts":               target_ts.isoformat(),
            "prediction_ts":           now_utc.isoformat(),
        })

df_preds = pd.DataFrame(rows)
print(f"Predictions: {len(df_preds)} rows ({df_preds['airport_code'].nunique()} airports × {df_preds['horizon_h'].nunique()} horizons)")
df_preds.head(10)

In [ ]:
# Write to two Delta tables. predictions_latest is overwritten so DirectLake always
# reads the most recent run; predictions_history is append-only for audit/trends.
write_deltalake(PREDICTIONS_LATEST,  df_preds, mode="overwrite", schema_mode="overwrite")
write_deltalake(PREDICTIONS_HISTORY, df_preds, mode="append")

print(f"predictions_latest:  {len(df_preds)} rows (overwrite)")
print(f"predictions_history: {len(df_preds)} rows appended")

In [ ]:
# Verification — confirms exactly 350 rows, 70 distinct airports, plausible delay range.
df_check = DeltaTable(PREDICTIONS_LATEST).to_pandas()

print(f"predictions_latest: {len(df_check)} rows")
print(f"  airports: {df_check['airport_code'].nunique()}")
print(f"  horizons: {sorted(df_check['horizon_h'].unique().tolist())}")
print(f"  prediction_ts: {df_check['prediction_ts'].iloc[0]}")
print()
print("Delay distribution by horizon (minutes):")
print(df_check.groupby("horizon_h")["predicted_arr_delay_min"].agg(["mean", "min", "max"]).round(2))